# Fine-Tune Last Layer — Consistency Impact

Investigates whether fine-tuning only the final FC layer improves decision consistency.
- Freeze all layers except last FC
- Fine-tune on CIFAR-10 training data (5k samples, 5 epochs)
- Compare CS before and after fine-tuning

**Update DATA_DIR before running.**

In [ ]:
DATA_DIR   = r'E:\cifar-10-python'
OUTPUT_DIR = r'E:\decision_consistency_finetune'
NUM_FINETUNE=5000; NUM_EVAL=500; EPOCHS=5; LR=1e-3; SEED=42; ALPHA=0.5

In [ ]:
import os, io, gc, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)

In [ ]:
preprocess=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
cifar_train=datasets.CIFAR10(root=DATA_DIR,train=True,transform=preprocess,download=False)
cifar_test=datasets.CIFAR10(root=DATA_DIR,train=False,transform=preprocess,download=False)
print(f'Train: {len(cifar_train)}  Test: {len(cifar_test)}')

In [ ]:
ft_model=models.resnet18(pretrained=True)
for param in ft_model.parameters(): param.requires_grad=False
ft_model.fc=nn.Linear(ft_model.fc.in_features,10)
ft_model=ft_model.to(device)
trainable=sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
print(f'Trainable params (last layer only): {trainable:,}')

In [ ]:
idx_ft=np.random.choice(len(cifar_train),NUM_FINETUNE,replace=False)
train_sub=torch.utils.data.Subset(cifar_train,idx_ft); train_dl=DataLoader(train_sub,batch_size=64,shuffle=True)
optimizer=torch.optim.Adam(ft_model.fc.parameters(),lr=LR); criterion=nn.CrossEntropyLoss()
ft_model.train()
for epoch in range(EPOCHS):
    total_loss,correct,total=0,0,0
    for X_batch,y_batch in tqdm(train_dl,desc=f'Epoch {epoch+1}/{EPOCHS}',leave=False):
        X_batch,y_batch=X_batch.to(device),y_batch.to(device)
        optimizer.zero_grad(); out=ft_model(X_batch); loss=criterion(out,y_batch)
        loss.backward(); optimizer.step(); total_loss+=loss.item()*len(y_batch); correct+=(out.argmax(1)==y_batch).sum().item(); total+=len(y_batch)
    print(f'Epoch {epoch+1}: loss={total_loss/total:.4f}  acc={correct/total:.3f}')
ft_model.eval(); print('Fine-tuning complete.')

In [ ]:
def apply_jpeg(t,q=75):
    buf=io.BytesIO(); transforms.ToPILImage()(t.cpu().clamp(0,1)).save(buf,format='JPEG',quality=q); buf.seek(0); return transforms.ToTensor()(Image.open(buf))
def generate_perturbations(t):
    noise=0.01*torch.randn_like(t)
    return {'rotation':TF.rotate(t,angle=2),'brightness':TF.adjust_brightness(t,brightness_factor=1.1),'noise':torch.clamp(t+noise,0,1),'contrast':TF.adjust_contrast(t,contrast_factor=1.1),'blur':TF.gaussian_blur(t,kernel_size=3,sigma=0.5),'jpeg':apply_jpeg(t)}
def eval_cs(model,test_dl,n,dev,alpha=ALPHA):
    model.eval(); metrics=[]; count=0
    for X_batch,y_batch in test_dl:
        for img in X_batch:
            if count>=n: break
            img=img.to(dev)
            with torch.no_grad(): probs=F.softmax(model(img.unsqueeze(0)),dim=1)
            op=probs.argmax(1).item(); oc=probs.max().item()
            pr={}
            for k,v in generate_perturbations(img.cpu()).items():
                with torch.no_grad(): pb=F.softmax(model(v.unsqueeze(0).to(dev)),dim=1)
                pr[k]=(pb.argmax(1).item(),pb.max().item())
            K=len(pr); S=sum(1 for p,_ in pr.values() if p==op)/K; D=sum(abs(c-oc)/max(oc,1-oc) for _,c in pr.values())/K
            metrics.append({'LS':S,'CD':D,'CS':alpha*S+(1-alpha)*(1-D)}); count+=1
        if count>=n: break
    return metrics
test_dl_small=DataLoader(torch.utils.data.Subset(cifar_test,range(NUM_EVAL)),batch_size=32)
pretrained=models.resnet18(pretrained=True).eval().to(device)
m_pre=eval_cs(pretrained,test_dl_small,NUM_EVAL,device)
m_ft=eval_cs(ft_model,test_dl_small,NUM_EVAL,device)
print(f'\n{"":<12} {"Pretrained":>12} {"Fine-tuned":>12}')
for metric in ['LS','CD','CS']:
    pre_m=np.mean([m[metric] for m in m_pre]); ft_m=np.mean([m[metric] for m in m_ft])
    print(f'{metric:<12} {pre_m:>12.3f} {ft_m:>12.3f}')

In [ ]:
cs_pre=[m['CS'] for m in m_pre]; cs_ft=[m['CS'] for m in m_ft]
fig,axes=plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax,vals,title in [(axes[0],cs_pre,f'Pretrained  Mean={np.mean(cs_pre):.3f}'),(axes[1],cs_ft,f'Fine-tuned last layer  Mean={np.mean(cs_ft):.3f}')]:
    ax.hist(vals,bins=20,color='steelblue',edgecolor='white'); ax.axvline(np.mean(vals),color='red',linestyle='--',lw=1.5); ax.set_xlabel('CS'); ax.set_title(title,fontsize=10); ax.set_xlim(0,1); ax.grid(alpha=0.3)
axes[0].set_ylabel('Frequency'); fig.suptitle('Effect of Last-Layer Fine-Tuning on Decision Consistency'); plt.tight_layout()
path=os.path.join(OUTPUT_DIR,'finetune_vs_pretrained_cs.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)